# Chapter 2.1: Logistic Regression & FTRL-Proximal for CTR Prediction

## Learning Objectives

By the end of this notebook, you will be able to:

1. Explain why logistic regression is a strong baseline for CTR prediction
2. Implement the **feature hashing trick** to handle high-dimensional sparse features
3. Derive and implement the **FTRL-Proximal** optimization algorithm from scratch
4. Understand **per-coordinate learning rates** and why they matter for sparse data
5. Apply **L1/L2 regularization** to produce sparse, memory-efficient models
6. Compare FTRL-Proximal with standard SGD on synthetic CTR data
7. Evaluate CTR models using log-loss, AUC, and calibration metrics

## Prerequisites

- Basic understanding of logistic regression and gradient descent
- Familiarity with Python, NumPy, and matplotlib
- Part 1 of this course (Foundations of Recommendation Systems)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/USERNAME/rec_system/blob/main/notebooks/part2/chapter_2.1_lr_ftrl.ipynb)
[![Download](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/USERNAME/rec_system/main/notebooks/part2/chapter_2.1_lr_ftrl.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, log_loss
from collections import defaultdict
import hashlib

np.random.seed(42)

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## 1. Why Logistic Regression for CTR?

Click-Through Rate (CTR) prediction is a **binary classification** problem: given a user, an item, and a context, will the user click?

Logistic Regression (LR) has been the **industry workhorse** for CTR prediction for years, used at Google, Facebook, and many other companies. Why?

- **Scalability**: Handles billions of features and training examples
- **Interpretability**: Feature weights tell us what drives clicks
- **Online learning**: Can update incrementally as new data arrives
- **Sparsity**: With L1 regularization, most weights become exactly zero

The model predicts:

\[ P(y=1 | \mathbf{x}) = \sigma(\mathbf{w}^T \mathbf{x}) = \frac{1}{1 + e^{-\mathbf{w}^T \mathbf{x}}} \]

where \(\mathbf{x}\) is typically a very high-dimensional, sparse feature vector (one-hot encoded categorical features).

## 2. Generating Synthetic CTR Data

We'll create a synthetic CTR dataset that mimics real-world ad click data with:
- User features (age group, gender, etc.)
- Item features (ad category, advertiser, etc.)
- Context features (time of day, device, etc.)

In [ ]:
def generate_ctr_data(n_samples=50000, seed=42):
    """Generate synthetic CTR data with categorical features."""
    rng = np.random.RandomState(seed)
    
    # Categorical features
    user_age = rng.choice(['18-24', '25-34', '35-44', '45-54', '55+'], n_samples)
    user_gender = rng.choice(['M', 'F'], n_samples)
    ad_category = rng.choice(['tech', 'fashion', 'food', 'travel', 'finance', 'health'], n_samples)
    advertiser = rng.choice([f'adv_{i}' for i in range(20)], n_samples)
    device = rng.choice(['mobile', 'desktop', 'tablet'], n_samples)
    hour = rng.choice([f'h_{i}' for i in range(24)], n_samples)
    
    # Create feature dictionaries for each sample
    features = []
    for i in range(n_samples):
        feat = {
            'user_age': user_age[i],
            'user_gender': user_gender[i],
            'ad_category': ad_category[i],
            'advertiser': advertiser[i],
            'device': device[i],
            'hour': hour[i],
        }
        features.append(feat)
    
    # Generate labels with a realistic pattern
    base_ctr = 0.05  # 5% base CTR
    logits = np.zeros(n_samples)
    
    # Age effects
    age_effect = {'18-24': 0.3, '25-34': 0.2, '35-44': 0.0, '45-54': -0.1, '55+': -0.3}
    for i in range(n_samples):
        logits[i] += age_effect[user_age[i]]
    
    # Category effects
    cat_effect = {'tech': 0.4, 'fashion': 0.2, 'food': 0.1, 'travel': -0.1, 'finance': -0.2, 'health': 0.0}
    for i in range(n_samples):
        logits[i] += cat_effect[ad_category[i]]
    
    # Device effect
    dev_effect = {'mobile': 0.2, 'desktop': 0.0, 'tablet': -0.1}
    for i in range(n_samples):
        logits[i] += dev_effect[device[i]]
    
    # Interaction: young users + tech ads => higher CTR
    for i in range(n_samples):
        if user_age[i] in ['18-24', '25-34'] and ad_category[i] == 'tech':
            logits[i] += 0.5
    
    logits += np.log(base_ctr / (1 - base_ctr))  # shift to get ~5% base CTR
    probs = 1.0 / (1.0 + np.exp(-logits))
    labels = rng.binomial(1, probs)
    
    print(f"Generated {n_samples} samples, CTR = {labels.mean():.4f}")
    return features, labels

features, labels = generate_ctr_data()

# Train/test split
split = 40000
train_features, test_features = features[:split], features[split:]
train_labels, test_labels = labels[:split], labels[split:]

## 3. Feature Hashing (The Hashing Trick)

In production CTR systems, the feature space can have **billions** of dimensions (e.g., all possible user-ad pairs). Maintaining a feature dictionary is impractical.

The **hashing trick** (Weinberger et al., 2009) maps features to a fixed-size vector using a hash function:

\[ \phi(x) = \text{hash}(\text{feature\_name}=\text{feature\_value}) \mod D \]

where \(D\) is the hash space size.

> **Concept:** The hashing trick trades a small amount of accuracy (due to hash collisions) for massive memory savings. In practice with \(D = 2^{20}\) to \(2^{24}\), collision effects are negligible.

> **Common Pitfall:** Don't use too small a hash space. With too many collisions, unrelated features interfere with each other, degrading model quality.

In [ ]:
class FeatureHasher:
    """Feature hashing (hashing trick) for sparse categorical features."""
    
    def __init__(self, n_features=2**16):
        self.n_features = n_features
    
    def hash_feature(self, name, value):
        """Hash a feature name-value pair to an index."""
        raw = f"{name}={value}".encode('utf-8')
        h = int(hashlib.md5(raw).hexdigest(), 16)
        return h % self.n_features
    
    def transform(self, feature_dict):
        """Transform a feature dictionary to a sparse representation (list of indices)."""
        indices = []
        for name, value in feature_dict.items():
            idx = self.hash_feature(name, value)
            indices.append(idx)
        return sorted(set(indices))  # deduplicate

hasher = FeatureHasher(n_features=2**16)

# Example
sample = train_features[0]
print(f"Original features: {sample}")
print(f"Hashed indices: {hasher.transform(sample)}")
print(f"Hash space size: {hasher.n_features}")

## 4. Logistic Regression with SGD (Baseline)

Before implementing FTRL, let's build a simple LR model trained with SGD as our baseline.

For sparse data, we only update weights for **active features** (non-zero entries):

\[ w_i \leftarrow w_i - \eta \cdot (\hat{y} - y) \cdot x_i \]

where \(\hat{y} = \sigma(\mathbf{w}^T \mathbf{x})\).

In [ ]:
class LogisticRegressionSGD:
    """Logistic Regression with SGD for sparse features."""
    
    def __init__(self, n_features, lr=0.1, l2=1e-5):
        self.w = np.zeros(n_features)
        self.bias = 0.0
        self.lr = lr
        self.l2 = l2
    
    def predict(self, indices):
        """Predict probability for a single sample (sparse indices)."""
        logit = self.bias + np.sum(self.w[indices])
        return 1.0 / (1.0 + np.exp(-np.clip(logit, -35, 35)))
    
    def update(self, indices, label):
        """Update weights for a single sample."""
        pred = self.predict(indices)
        error = pred - label
        
        # Update bias
        self.bias -= self.lr * error
        
        # Update active weights with L2 regularization
        for idx in indices:
            self.w[idx] -= self.lr * (error + self.l2 * self.w[idx])
    
    def train_epoch(self, features_list, labels, hasher):
        """Train one epoch over the dataset."""
        indices = np.arange(len(labels))
        np.random.shuffle(indices)
        
        total_loss = 0.0
        for i in indices:
            feat_indices = hasher.transform(features_list[i])
            pred = self.predict(feat_indices)
            # Log loss for this sample
            pred_clipped = np.clip(pred, 1e-7, 1 - 1e-7)
            total_loss += -(labels[i] * np.log(pred_clipped) + (1 - labels[i]) * np.log(1 - pred_clipped))
            self.update(feat_indices, labels[i])
        
        return total_loss / len(labels)

# Train SGD baseline
sgd_model = LogisticRegressionSGD(n_features=hasher.n_features, lr=0.1, l2=1e-5)
sgd_losses = []

for epoch in range(5):
    loss = sgd_model.train_epoch(train_features, train_labels, hasher)
    sgd_losses.append(loss)
    
    # Evaluate on test set
    preds = [sgd_model.predict(hasher.transform(f)) for f in test_features]
    auc = roc_auc_score(test_labels, preds)
    test_loss = log_loss(test_labels, preds)
    print(f"Epoch {epoch+1}: train_loss={loss:.4f}, test_loss={test_loss:.4f}, AUC={auc:.4f}")

## 5. FTRL-Proximal Algorithm

FTRL-Proximal (Follow-The-Regularized-Leader) was introduced by Google in their landmark paper *"Ad Click Prediction: a View from the Trenches"* (McMahan et al., 2013). It became the standard algorithm for large-scale CTR prediction at Google.

### Why FTRL over SGD?

1. **Better sparsity**: FTRL with L1 regularization produces truly sparse models (exact zeros), while SGD with L1 rarely achieves exact sparsity
2. **Per-coordinate learning rates**: Frequent features get smaller learning rates; rare features retain larger ones
3. **Online learning**: Naturally processes one example at a time

### The Algorithm

For each coordinate \(i\), FTRL maintains:
- \(z_i\): accumulated gradient information
- \(n_i\): sum of squared gradients (for adaptive learning rate)

The weight update rule is:

\[ w_i = \begin{cases} 0 & \text{if } |z_i| \leq \lambda_1 \\ -\frac{1}{\frac{\beta + \sqrt{n_i}}{\alpha} + \lambda_2} \left(z_i - \text{sign}(z_i) \cdot \lambda_1\right) & \text{otherwise} \end{cases} \]

where:
- \(\alpha\) controls the learning rate schedule
- \(\beta\) ensures numerical stability (typically 1.0)
- \(\lambda_1\) is the L1 regularization strength
- \(\lambda_2\) is the L2 regularization strength

The per-coordinate learning rate is: \(\eta_i = \frac{\alpha}{\beta + \sqrt{n_i}}\)

> **Concept:** The key insight is that FTRL performs the L1 proximal step *after* accumulating gradients, which produces much sparser solutions than applying L1 penalty in the gradient update (as in SGD).

> **Pro Tip:** In production, the sparsity from FTRL is critical for model serving. A model with 10 billion features but only 10 million non-zero weights fits in memory and is fast to evaluate.

In [ ]:
class FTRLProximal:
    """FTRL-Proximal optimizer for logistic regression (McMahan et al., 2013)."""
    
    def __init__(self, n_features, alpha=0.5, beta=1.0, lambda1=0.1, lambda2=1.0):
        self.n_features = n_features
        self.alpha = alpha       # Learning rate parameter
        self.beta = beta         # Smoothing parameter
        self.lambda1 = lambda1   # L1 regularization
        self.lambda2 = lambda2   # L2 regularization
        
        # Per-coordinate accumulators
        self.z = np.zeros(n_features)  # Accumulated gradient info
        self.n = np.zeros(n_features)  # Sum of squared gradients
        
        # Bias terms
        self.z_bias = 0.0
        self.n_bias = 0.0
    
    def _get_weight(self, i):
        """Compute weight for coordinate i using FTRL update rule."""
        if abs(self.z[i]) <= self.lambda1:
            return 0.0
        else:
            sign = 1.0 if self.z[i] >= 0 else -1.0
            w = -(self.z[i] - sign * self.lambda1) / (
                (self.beta + np.sqrt(self.n[i])) / self.alpha + self.lambda2
            )
            return w
    
    def _get_bias(self):
        """Compute bias (no regularization on bias)."""
        if self.n_bias == 0:
            return 0.0
        return -self.z_bias / ((self.beta + np.sqrt(self.n_bias)) / self.alpha)
    
    def predict(self, indices):
        """Predict probability for a single sample."""
        logit = self._get_bias()
        for i in indices:
            logit += self._get_weight(i)
        return 1.0 / (1.0 + np.exp(-np.clip(logit, -35, 35)))
    
    def update(self, indices, label):
        """Update accumulators for a single sample."""
        pred = self.predict(indices)
        grad = pred - label  # gradient of log loss
        
        # Update bias
        sigma_bias = (np.sqrt(self.n_bias + grad**2) - np.sqrt(self.n_bias)) / self.alpha
        self.z_bias += grad - sigma_bias * self._get_bias()
        self.n_bias += grad**2
        
        # Update each active coordinate
        for i in indices:
            w_i = self._get_weight(i)
            sigma_i = (np.sqrt(self.n[i] + grad**2) - np.sqrt(self.n[i])) / self.alpha
            self.z[i] += grad - sigma_i * w_i
            self.n[i] += grad**2
        
        return pred
    
    def train_epoch(self, features_list, labels, hasher):
        """Train one epoch."""
        indices = np.arange(len(labels))
        np.random.shuffle(indices)
        
        total_loss = 0.0
        for i in indices:
            feat_indices = hasher.transform(features_list[i])
            pred = self.update(feat_indices, labels[i])
            pred_clipped = np.clip(pred, 1e-7, 1 - 1e-7)
            total_loss += -(labels[i] * np.log(pred_clipped) + 
                           (1 - labels[i]) * np.log(1 - pred_clipped))
        
        return total_loss / len(labels)
    
    def count_nonzero(self):
        """Count non-zero weights (sparsity measure)."""
        count = 0
        for i in range(self.n_features):
            if self._get_weight(i) != 0.0:
                count += 1
        return count

print("FTRL-Proximal class defined successfully.")

In [ ]:
# Train FTRL-Proximal
ftrl_model = FTRLProximal(n_features=hasher.n_features, alpha=0.5, beta=1.0, 
                           lambda1=0.1, lambda2=1.0)
ftrl_losses = []
ftrl_aucs = []

for epoch in range(5):
    loss = ftrl_model.train_epoch(train_features, train_labels, hasher)
    ftrl_losses.append(loss)
    
    # Evaluate
    preds = [ftrl_model.predict(hasher.transform(f)) for f in test_features]
    auc = roc_auc_score(test_labels, preds)
    test_loss = log_loss(test_labels, preds)
    ftrl_aucs.append(auc)
    print(f"Epoch {epoch+1}: train_loss={loss:.4f}, test_loss={test_loss:.4f}, AUC={auc:.4f}")

## 6. Comparing SGD vs FTRL: Sparsity and Convergence

In [ ]:
# Compare training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
axes[0].plot(range(1, 6), sgd_losses, 'b-o', label='SGD', linewidth=2)
axes[0].plot(range(1, 6), ftrl_losses, 'r-s', label='FTRL-Proximal', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Training Log Loss')
axes[0].set_title('Training Loss: SGD vs FTRL-Proximal')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Sparsity comparison
sgd_nonzero = np.count_nonzero(sgd_model.w)
# Sample a subset to count FTRL non-zeros (full count is slow for 2^16)
ftrl_nonzero = sum(1 for i in range(hasher.n_features) if ftrl_model._get_weight(i) != 0.0)

methods = ['SGD', 'FTRL-Proximal']
nonzeros = [sgd_nonzero, ftrl_nonzero]
colors = ['steelblue', 'coral']

axes[1].bar(methods, nonzeros, color=colors, edgecolor='black')
axes[1].set_ylabel('Number of Non-Zero Weights')
axes[1].set_title('Model Sparsity Comparison')
for i, v in enumerate(nonzeros):
    axes[1].text(i, v + 50, str(v), ha='center', fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"SGD non-zero weights: {sgd_nonzero} / {hasher.n_features}")
print(f"FTRL non-zero weights: {ftrl_nonzero} / {hasher.n_features}")
print(f"FTRL sparsity: {1 - ftrl_nonzero/hasher.n_features:.4%}")

## 7. Per-Coordinate Learning Rates

A key innovation in FTRL is the **per-coordinate learning rate**:

\[ \eta_{t,i} = \frac{\alpha}{\beta + \sqrt{\sum_{s=1}^{t} g_{s,i}^2}} \]

This is similar to AdaGrad's adaptive learning rate. Features that appear frequently accumulate more squared gradients, resulting in smaller learning rates. Rare features retain higher learning rates.

> **Concept:** This is crucial for CTR prediction because feature frequencies are highly skewed (power-law). A popular ad might appear millions of times, while a niche ad appears only a few times. Per-coordinate rates let the model learn quickly from rare features while not over-fitting to common ones.

In [ ]:
# Visualize per-coordinate learning rates
# Get learning rates for all non-zero features
active_indices = [i for i in range(hasher.n_features) if ftrl_model.n[i] > 0]
learning_rates = [ftrl_model.alpha / (ftrl_model.beta + np.sqrt(ftrl_model.n[i])) 
                  for i in active_indices]
grad_sums = [ftrl_model.n[i] for i in active_indices]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(learning_rates, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Learning Rate')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Per-Coordinate Learning Rates')
axes[0].grid(True, alpha=0.3)

axes[1].scatter(grad_sums, learning_rates, alpha=0.5, s=10, c='coral')
axes[1].set_xlabel('Accumulated Squared Gradients (n_i)')
axes[1].set_ylabel('Learning Rate')
axes[1].set_title('Learning Rate vs Feature Frequency')
axes[1].set_xscale('log')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Effect of L1 Regularization Strength

Let's study how the L1 regularization parameter \(\lambda_1\) controls the sparsity-accuracy tradeoff.

In [ ]:
lambda1_values = [0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0]
results = []

for l1 in lambda1_values:
    model = FTRLProximal(n_features=hasher.n_features, alpha=0.5, beta=1.0,
                         lambda1=l1, lambda2=1.0)
    
    for epoch in range(3):  # Quick training
        model.train_epoch(train_features, train_labels, hasher)
    
    preds = [model.predict(hasher.transform(f)) for f in test_features]
    auc = roc_auc_score(test_labels, preds)
    n_nonzero = sum(1 for i in range(hasher.n_features) if model._get_weight(i) != 0.0)
    
    results.append({'lambda1': l1, 'auc': auc, 'nonzero': n_nonzero})
    print(f"lambda1={l1:.3f}: AUC={auc:.4f}, non-zero={n_nonzero}")

# Plot
fig, ax1 = plt.subplots(figsize=(10, 6))

l1s = [r['lambda1'] for r in results]
aucs = [r['auc'] for r in results]
nzs = [r['nonzero'] for r in results]

ax1.plot(l1s, aucs, 'b-o', linewidth=2, markersize=8, label='AUC')
ax1.set_xlabel('L1 Regularization (lambda1)', fontsize=12)
ax1.set_ylabel('AUC', color='blue', fontsize=12)
ax1.tick_params(axis='y', labelcolor='blue')
ax1.set_xscale('log')

ax2 = ax1.twinx()
ax2.plot(l1s, nzs, 'r-s', linewidth=2, markersize=8, label='Non-zero weights')
ax2.set_ylabel('Non-Zero Weights', color='red', fontsize=12)
ax2.tick_params(axis='y', labelcolor='red')

plt.title('L1 Regularization: Sparsity vs Accuracy Tradeoff', fontsize=14)
fig.legend(loc='upper center', bbox_to_anchor=(0.5, 0.92), ncol=2)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Online Learning Simulation

One of FTRL's strengths is **online learning** -- processing one sample at a time without storing the entire dataset. Let's simulate a streaming scenario.

In [ ]:
# Simulate online learning: track metrics over time
online_model = FTRLProximal(n_features=hasher.n_features, alpha=0.5, beta=1.0,
                             lambda1=0.1, lambda2=1.0)

window_size = 1000
running_loss = []
running_auc = []
window_preds = []
window_labels = []

for i in range(len(train_features)):
    feat_idx = hasher.transform(train_features[i])
    pred = online_model.predict(feat_idx)
    
    window_preds.append(pred)
    window_labels.append(train_labels[i])
    
    online_model.update(feat_idx, train_labels[i])
    
    if (i + 1) % window_size == 0:
        wp = np.array(window_preds[-window_size:])
        wl = np.array(window_labels[-window_size:])
        wp_clipped = np.clip(wp, 1e-7, 1 - 1e-7)
        wloss = -np.mean(wl * np.log(wp_clipped) + (1 - wl) * np.log(1 - wp_clipped))
        running_loss.append(wloss)
        if len(np.unique(wl)) > 1:
            running_auc.append(roc_auc_score(wl, wp))
        else:
            running_auc.append(0.5)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x_ticks = [(i+1) * window_size for i in range(len(running_loss))]

axes[0].plot(x_ticks, running_loss, 'b-', linewidth=1.5, alpha=0.8)
axes[0].set_xlabel('Samples Processed')
axes[0].set_ylabel('Log Loss (rolling window)')
axes[0].set_title('Online Learning: Loss Over Time')
axes[0].grid(True, alpha=0.3)

axes[1].plot(x_ticks, running_auc, 'r-', linewidth=1.5, alpha=0.8)
axes[1].set_xlabel('Samples Processed')
axes[1].set_ylabel('AUC (rolling window)')
axes[1].set_title('Online Learning: AUC Over Time')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Exercises

### Exercise 1: Implement FTRL with Different Hyperparameters

Experiment with the `alpha` and `beta` parameters to see their effect on convergence speed and final performance.

In [ ]:
# Exercise 1: Compare different alpha values
# TODO: Train FTRL models with alpha in [0.05, 0.1, 0.5, 1.0, 5.0]
# TODO: Plot AUC vs alpha
# TODO: Which alpha gives the best AUC? Why?

alpha_values = [0.05, 0.1, 0.5, 1.0, 5.0]
# YOUR CODE HERE
# for alpha in alpha_values:
#     model = FTRLProximal(n_features=hasher.n_features, alpha=alpha, ...)
#     ...

### Exercise 2: Feature Hashing Collision Analysis

Investigate how the hash space size affects model quality due to collisions.

In [ ]:
# Exercise 2: Vary hash space size and measure AUC
# TODO: Try hash space sizes: [2**8, 2**10, 2**12, 2**14, 2**16, 2**18]
# TODO: For each, train FTRL and measure test AUC
# TODO: Plot hash space size vs AUC -- at what point do collisions stop mattering?

hash_sizes = [2**8, 2**10, 2**12, 2**14, 2**16, 2**18]
# YOUR CODE HERE

### Exercise 3: Cross Features

Feature crosses (conjunctions) are key to LR-based CTR models. Implement cross features and measure their impact.

In [ ]:
# Exercise 3: Add cross features to the feature hasher
# TODO: Extend FeatureHasher to add pairwise cross features
#       e.g., "user_age=18-24 X ad_category=tech"
# TODO: Train FTRL with and without cross features
# TODO: Compare AUC -- cross features should improve the model

class FeatureHasherWithCrosses(FeatureHasher):
    def __init__(self, n_features=2**16, cross_fields=None):
        super().__init__(n_features)
        # cross_fields: list of tuples, e.g., [('user_age', 'ad_category'), ...]
        self.cross_fields = cross_fields or []
    
    def transform(self, feature_dict):
        indices = super().transform(feature_dict)
        # TODO: Add cross feature hashing
        # YOUR CODE HERE
        return sorted(set(indices))

# YOUR CODE HERE

### Exercise 4: SGD vs FTRL Online Learning Race

Run both SGD and FTRL in online mode on the same data stream and compare their convergence.

In [ ]:
# Exercise 4: Online learning comparison
# TODO: Run SGD and FTRL simultaneously on the same shuffled data stream
# TODO: Track rolling-window AUC for both
# TODO: Plot both curves on the same chart
# TODO: Which converges faster? Which achieves a sparser model?

# YOUR CODE HERE

## Summary

In this chapter, we covered:

1. **Logistic Regression** as the foundation of industrial CTR prediction
2. **Feature hashing** for handling high-dimensional sparse features efficiently
3. **FTRL-Proximal** (Google, 2013): the workhorse optimizer for sparse CTR models
   - Per-coordinate learning rates adapt to feature frequency
   - L1 proximal step produces truly sparse weights
4. **Online learning** enables incremental model updates in streaming settings

### Key Takeaways

- FTRL achieves much better sparsity than SGD while maintaining similar accuracy
- Per-coordinate learning rates are essential for skewed feature distributions
- Feature hashing eliminates the need for a feature dictionary
- Cross features are critical for capturing interactions in linear models

### What's Next

In Chapter 2.2, we'll move beyond manual feature crossing with **Factorization Machines**, which automatically learn pairwise feature interactions through latent vectors.